In [2]:
from transformers import (
    AutoTokenizer,
    pipeline,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import pandas as pd

In [3]:
from datasets import Dataset

MODEL_NAME = "roberta-base"
DATA_DIR = "/kaggle/input/datasets/ababeiraul/tweets"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)

print(test_df.head(1))

   id keyword location                                text
0   0     NaN      NaN  Just happened a terrible car crash


In [29]:
print(len(test_df))

3263


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token = tokenizer.tokenize(hf_train["text"][0])

print(token)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

['Our', 'ĠDe', 'eds', 'Ġare', 'Ġthe', 'ĠReason', 'Ġof', 'Ġthis', 'Ġ#', 'earth', 'qu', 'ake', 'ĠMay', 'ĠALL', 'AH', 'ĠFor', 'give', 'Ġus', 'Ġall']


In [6]:
id2label = {
    0: "REAL",
    1: "FAKE"
}

label2id = {
    "REAL": 0,
    "FAKE": 1
}

def preprocess_text(examples):
    return tokenizer([" ".join(x) for x in examples["text"]])

tokenized_df_train = hf_train.map(
    preprocess_text,
    batched = True,
    num_proc = 4
)

tokenized_df_test = hf_test.map(
    preprocess_text,
    batched = True,
    num_proc = 4
)

Map (num_proc=4):   0%|          | 0/7613 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3263 [00:00<?, ? examples/s]

In [30]:
def true_process_function(examples):
    return tokenizer(
        examples["text"],
        truncation = True,
        padding=False,
        max_length=128
    )

tokenized_train = tokenized_df_train.map(
    true_process_function,
    batched = True,
    num_proc = 4
)

tokenized_test = tokenized_df_test.map(
    true_process_function,
    batched = True,
    num_proc = 4
)

Map (num_proc=4):   0%|          | 0/7613 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3263 [00:00<?, ? examples/s]

In [32]:
# tokenized_test = tokenized_test.rename_column("target", "labels")
tokenized_train = tokenized_train.rename_column("target", "labels")

In [33]:
print(tokenized_test.column_names)

['id', 'keyword', 'location', 'text', 'input_ids', 'attention_mask']


In [34]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [35]:
!pip install -q evaluate

In [36]:
import evaluate 
import numpy as np

f1_score = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = 1)

    f1 = f1_score.compute(
        predictions=predictions,
        references=labels,
        average="weighted" # accounts imbalance in the dataset
    )

    accuracy = accuracy_metric.compute(
        predictions = predictions,
        references = labels,
    )

    return {
        "f1": f1["f1"],
        "accuracy": accuracy["accuracy"]
    }




In [37]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/tweets_model",

    # base hyperparams
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16 = True,
    weight_decay = 0.01,

    # evaluation and saving
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1, # only save one checkpoint
    load_best_model_at_end = True, # gives us the best checkpoint of the model
    metric_for_best_model = "f1",

    # logging
    logging_steps = 50, # log every 50 steps
    report_to = "none"
)

In [39]:
split = tokenized_train.train_test_split(test_size = 0.1, seed = 42)

In [40]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = split["train"],
    eval_dataset = split["test"],
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [41]:
train_results = trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.759877,0.792072,0.834927,0.835958
2,0.656106,0.758652,0.838086,0.841207
3,0.559593,0.863901,0.822645,0.822835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
print(train_results)

TrainOutput(global_step=645, training_loss=0.7096170292344204, metrics={'train_runtime': 190.0422, 'train_samples_per_second': 108.15, 'train_steps_per_second': 3.394, 'total_flos': 624138992749980.0, 'train_loss': 0.7096170292344204, 'epoch': 3.0})


In [44]:

raw_predicts = trainer.predict(tokenized_test);

predicted_classes = np.argmax(raw_predicts.predictions, axis = 1)

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": predicted_classes
})

submission.to_csv("submission.csv", index = False)

print(submission.head())

   id  target
0   0       1
1   2       1
2   3       1
3   9       1
4  11       1


In [43]:
print(len(submission))

3263
